In [32]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as functions
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

from pyspark.sql.functions import col, count, when

In [23]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/Users/alanyom/Desktop/pyspark_ws/data.csv")

df.printSchema()
df.show(5)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)

+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|InvoiceNo|StockCode|         Description|Quantity|   InvoiceDate|UnitPrice|CustomerID|       Country|
+---------+---------+--------------------+--------+--------------+---------+----------+--------------+
|   536365|   85123A|WHITE HANGING HEA...|       6|12/1/2010 8:26|     2.55|     17850|United Kingdom|
|   536365|    71053| WHITE METAL LANTERN|       6|12/1/2010 8:26|     3.39|     17850|United Kingdom|
|   536365|   84406B|CREAM CUPID HEART...|       8|12/1/2010 8:26|     2.75|     17850|United Kingdom|
|   536365|   84029G|KNITTED UNION FLA...|  

In [24]:
df.count()

541909

In [27]:
df.select("CustomerID").distinct().count()

4373

In [26]:
df.select("Country").distinct().count()

38

In [ ]:
df.describe("Quantity", "UnitPrice").show()

+-------+------------------+------------------+
|summary|          Quantity|         UnitPrice|
+-------+------------------+------------------+
|  count|            541909|            541909|
|   mean|  9.55224954743324|4.6111136260897085|
| stddev|218.08115785023438| 96.75985306117963|
|    min|            -80995|         -11062.06|
|    max|             80995|           38970.0|
+-------+------------------+------------------+



In [29]:
df.filter(F.col("Quantity") <= 0).count()

10624

In [30]:
df.filter(F.col("UnitPrice") <= 0).count()

2517

In [31]:
df_clean = df.filter(F.col("Quantity") > 0).filter(F.col("UnitPrice") > 0)
df_clean.count()

530104

In [34]:
df_clean.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df_clean.columns
]).show()

+---------+---------+-----------+--------+-----------+---------+----------+-------+
|InvoiceNo|StockCode|Description|Quantity|InvoiceDate|UnitPrice|CustomerID|Country|
+---------+---------+-----------+--------+-----------+---------+----------+-------+
|        0|        0|          0|       0|          0|        0|    132220|      0|
+---------+---------+-----------+--------+-----------+---------+----------+-------+



In [39]:
df_clean_filled = df_clean.withColumn("CustomerID", F.col("CustomerID").cast("string"))
df_clean_filled = df_clean_filled.fillna({"CustomerID": "GUEST"})

In [40]:
df_clean_filled.filter(F.col("CustomerID") == "GUEST").count()

132220

In [ ]:
df_clean_filled.printSchema()

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: string (nullable = false)
 |-- Country: string (nullable = true)



In [42]:
df_clean_filled.select("InvoiceDate").show(5)

+--------------+
|   InvoiceDate|
+--------------+
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
|12/1/2010 8:26|
+--------------+
only showing top 5 rows

